# Conditional Image Generation

This tutorial is for readers who know basic Python and want to distinguish ways of controlling an image generator. By the end, you will generate from text, condition on pixels and image features, apply a LoRA, select using DreamSim embeddings, and sample a pretrained flow map.

Install `pip install -e ".[vision,notebooks]"` from the repository root. Full execution downloads pretrained weights and makes **four paid image API calls** on the first run. Use CUDA for TPIPS and the larger models; the examples release model memory between sections. Cached image requests can be reused, but metric inference is always real. There is no synthetic substitute for a missing model.

1. Text and native image conditioning with GPT-Image 2.5 and Nano Banana 2.
2. Stable Diffusion image-to-image, IP-Adapter, and LoRA.
3. DreamSim embedding selection.
4. Class-conditioned Decoupled MeanFlow.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
from dotenv import load_dotenv
ROOT = Path.cwd() if (Path.cwd() / "synthart").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env.local")
import numpy as np
import matplotlib.pyplot as plt
from synthart import ImageGenerator, Similarity, FlowMapGenerator, generate_similar
from synthart.images import release_memory
from synthart.plotting import gallery, plot_spaces
from synthart.experiment import cached_image
OUT = ROOT / "outputs" / "notebook-tour"
OUT.mkdir(parents=True, exist_ok=True)


## Text and Native Image Conditioning

A text prompt specifies desired content. A reference image supplies information that is hard to express in words. These are provider-native generation and editing calls; neither provider exposes a reproducible image seed here. GPT-Image 2.5 Flare is the default, with Sunburst selectable for editing. Nano Banana 2 uses `gemini-3.1-flash-image`; the newer Lite family member is also selectable, but Google recommends the full model for reference consistency.

The dated identifiers and official documentation are in [the literature guide](../docs/literature.md).

In [ ]:
prompt = "A delicate botanical ink drawing of a fern in a glass vase, muted sage wash on cream paper"
api_images = {}
for provider in ("openai", "gemini"):
    generator = ImageGenerator(provider)
    original = cached_image(generator, prompt, OUT / f"{provider}.png")
    edited = cached_image(generator, "Keep the drawing style; replace the glass vase with a ceramic bowl", OUT / f"{provider}-edit.png", reference=original)
    api_images[provider] = original
    gallery([original, edited], [f"{provider}: text", f"{provider}: image + text"], columns=2)
plt.show()

## Stable Diffusion and Image-to-Image

Image-to-image adds noise to an encoded reference and denoises it under the new prompt. Higher `strength` permits larger changes. SD 1.5 is a compact teaching baseline, not a claim about current generation quality. The same interface accepts an SDXL checkpoint. Fixed seeds improve repeatability within the same software and hardware environment.

In [ ]:
sd = ImageGenerator("diffusers")
reference = api_images["openai"]
base = cached_image(sd, prompt, OUT / "sd-text.png", seed=42, steps=20)
img2img = cached_image(sd, "A blue ceramic bowl holding a fern, botanical ink drawing", OUT / "sd-img2img.png", reference=reference, seed=42, steps=20, strength=0.65)
gallery([reference, base, img2img], ["Reference", "Text Only", "Image-to-Image"], columns=3)
plt.show()
del sd
release_memory()

## IP-Adapter

IP-Adapter encodes the reference into image features and adds learned image cross-attention to a frozen diffusion model. Its strength weights image conditioning. This is different from adding noise to the reference pixels, and its feature space is not DreamSim's space. A DreamSim vector cannot simply be inserted into the IP-Adapter input.

In [ ]:
adapter = ImageGenerator("diffusers", ip_adapter=True)
adapted = cached_image(adapter, "A fern in a ceramic bowl", OUT / "ip-adapter.png", reference=reference, seed=42, steps=20, strength=0.6)
gallery([reference, adapted], ["Reference", "IP-Adapter"], columns=2)
plt.show()
del adapter
release_memory()

## LoRA

A low-rank adapter changes a small set of learned weight updates. LoRA is an adaptation mechanism, not intrinsically a style model: it can learn style, subject identity, or faster sampling. Here the public LCM-LoRA supplies the last example and needs an LCM scheduler. Arbitrary style LoRAs can use `lora=repo_id`, `lora_weight=filename`, and `lora_scale=weight`; the adapter must match the base architecture and may require a trigger phrase. This tour performs inference, not adapter training.

The scheduler detail is wrapped by recognizing the LCM-LoRA repository below so the user-facing workflow remains a few lines.

In [ ]:
fast = ImageGenerator("diffusers", lora="latent-consistency/lcm-lora-sdv1-5")
fast_image = cached_image(fast, prompt, OUT / "lora.png", seed=42, steps=4, guidance=1.0)
gallery([base, fast_image], ["SD: 20 Steps", "LCM-LoRA: 4 Steps"], columns=2)
plt.show()
del fast
release_memory()

## DreamSim Embeddings as a Selection Condition

DreamSim maps each image into a human-aligned representation. We generate three candidates and select the smallest cosine distance to the reference embedding. This changes the distribution of accepted outputs without modifying the generator. It is **best-of-N selection**, not gradient guidance. The selection metric is optimistically biased when reused for evaluation; use held-out metrics or human judgments for validation.

In [ ]:
sd = ImageGenerator("diffusers")
metric = Similarity("dreamsim")
selected = generate_similar(sd, prompt, reference, metric=metric, candidates=3, seed=100, steps=20)
for i, image in enumerate(selected.candidates):
    image.save(OUT / f"dreamsim-candidate-{i}.png")
selected.image.save(OUT / "dreamsim-selected.png")
print({"distances": selected.distances, "selected_index": selected.index})
assert selected.distances[selected.index] <= min(selected.distances) + 1e-6
gallery([reference, *selected.candidates], ["Reference", *[f"Candidate {i}: {d:.3f}" for i, d in enumerate(selected.distances)]])
plt.show()
del sd, metric
release_memory()

## A Pretrained Flow Map

An ordinary flow-matching model predicts instantaneous velocity $v(x,t)$. Decoupled MeanFlow (DMF) predicts an interval-average velocity $u(x,t,r)$, allowing $x_r=x_t+(r-t)u(x_t,t,r)$ to span a large interval. The Hugging Face checkpoint below is class-conditioned on ImageNet and supports one or a few steps.

This is a genuine flow-map model, but **it is not the FMRG algorithm**. FMRG uses flow-map derivatives for reward guidance and its published FLUX setup is much heavier (the authors describe roughly 48 GB for 512px reward guidance). We retain a direct link and explain that tradeoff in the literature guide. The DMF inference wrapper uses PyTorch SDPA, eliminating upstream FlashAttention training extensions.

In [ ]:
flow = FlowMapGenerator()
one_step = cached_image(flow, "", OUT / "flow-1.png", class_id=207, seed=42, steps=1)
four_steps = cached_image(flow, "", OUT / "flow-4.png", class_id=207, seed=42, steps=4)
gallery([one_step, four_steps], ["DMF: One Step", "DMF: Four Steps"], columns=2)
plt.show()
del flow
release_memory()

## Exercise and Extensions

Which mechanism changes weights, which changes the denoising input, and which only changes which output is accepted? Fill in the mapping below, then compare with the answer.

ControlNet adds spatial conditions such as edges or depth; inpainting constrains an editable region; textual inversion learns a token representation; DreamBooth fine-tunes identity. These complement the mechanisms demonstrated here. See the primary links in the literature guide. Do not call a high similarity score proof of style transfer: content, palette, and layout can all drive it.

In [ ]:
mechanisms = {"LoRA": "learned weight updates", "image-to-image": "noised reference latent", "IP-Adapter": "image cross-attention", "DreamSim selection": "accepted output"}
mechanisms